In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch 
import numpy as np
import random
torch.autograd.set_detect_anomaly(True)
torch.multiprocessing.set_sharing_strategy("file_descriptor")
seed = 140421
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Register Calib

In [ ]:
from detectron2.data.datasets.pano360 import CalibDataset, CameraRegressorDataset

debug = False
train_calib = CalibDataset(
    train=True,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    debug=debug,
)

val_calib = CalibDataset(
    train=False,
    json_name="datasets/pano360_crops_dataset_cvpr_myDistWider_train.json",
    debug=debug,
)

train_pano = CameraRegressorDataset(
    is_train=True,
    debug=debug,
)
val_pano = CameraRegressorDataset(
    is_train=False,
    debug=debug,
)

In [ ]:
from detectron2.data import DatasetCatalog
DatasetCatalog.register("Pano360_scale_train", train_calib)
DatasetCatalog.register("Pano360_scale_val", val_calib)
DatasetCatalog.register("Pano360_train", train_pano)
DatasetCatalog.register("Pano360_val", val_pano)

# Register COCOScale

In [ ]:
from pathlib import Path

base_path = Path.cwd()
coco_path = base_path / "data" / "coco" 
coco_annotations_path = coco_path / "annotations" 
coco_keypoints_path = coco_annotations_path / "person_keypoints_train2017.json"
coco_keypoints_val_path = coco_annotations_path / "person_keypoints_val2017.json"
coco_scalenet_results_path = coco_path / "coco_results" 
coco_images_root_path =  coco_path / "train2017"
coco_val_images_root_path =  coco_path / "val2017"

In [ ]:
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets.builtin_meta import _get_builtin_metadata
from detectron2.data.datasets.coco_scale import COCOScale2017
coco_scale_train = COCOScale2017(
    debug=debug,
    camera_parameters_file_path=coco_scalenet_results_path / "yannick_results_train2017_filtered",
    coco_json_file_path=coco_keypoints_path,
    coco_image_root_path=coco_images_root_path,
    coco_scale_pickle_path=coco_scalenet_results_path / "results_with_kps_20200208_morethan2_2-8" / "pickle"
)
coco_meta = _get_builtin_metadata("coco_person")
coco_scale_dataset_name = "COCOScale2017_train"
DatasetCatalog.register(coco_scale_dataset_name, coco_scale_train)
coco_scale_meta = MetadataCatalog.get(coco_scale_dataset_name).set(
    json_file=coco_keypoints_path, image_root=coco_images_root_path, evaluator_type="coco", **coco_meta,
    thing_dataset_id_to_contiguous_id = {1: 0}  # COCO ID 1 → internal ID 0
)

In [ ]:
from detectron2.data.datasets.coco_scale import COCOScale2017
coco_scale_val = COCOScale2017(
    debug=debug,
    split="val",
    camera_parameters_file_path="",
    coco_json_file_path=coco_keypoints_val_path,
    coco_image_root_path=coco_val_images_root_path,
    coco_scale_pickle_path=coco_scalenet_results_path / "results_with_kps_20200225_val2017_test_detOnly_filtered_2-8_moreThan2" / "pickle",
)
coco_meta = _get_builtin_metadata("coco_person")
coco_scale_val_dataset_name = "COCOScale2017_val"
if coco_scale_val_dataset_name in DatasetCatalog:
    DatasetCatalog.remove(coco_scale_val_dataset_name)
DatasetCatalog.register(coco_scale_val_dataset_name, coco_scale_val)
coco_scale_meta = MetadataCatalog.get(coco_scale_val_dataset_name).set(
    json_file=coco_keypoints_val_path, image_root=coco_val_images_root_path, evaluator_type="coco", **coco_meta,
    thing_dataset_id_to_contiguous_id = {1: 0}  # COCO ID 1 → internal ID 0
)

In [ ]:
from detectron2.data.datasets.coco_scale import COCOScale2017
coco_scale_val = COCOScale2017(
    debug=debug,
    split="test",
    camera_parameters_file_path="",
    coco_json_file_path="",
    coco_image_root_path=coco_val_images_root_path,
    coco_scale_pickle_path=coco_scalenet_results_path / "results_test_20200302_Car_noSmall-ratio1-35-mergeWith-results_with_kps_20200225_train2017_detOnly_filtered_2-8_moreThan2" / "pickle",
)
coco_meta = _get_builtin_metadata("coco_person")
coco_scale_test_dataset_name = "COCOScale2017_test"
if coco_scale_test_dataset_name in DatasetCatalog:
    DatasetCatalog.remove(coco_scale_test_dataset_name)
DatasetCatalog.register(coco_scale_test_dataset_name, coco_scale_val)
coco_scale_meta = MetadataCatalog.get(coco_scale_test_dataset_name).set(
    json_file=coco_keypoints_val_path, image_root=coco_val_images_root_path, evaluator_type="coco", **coco_meta,
    thing_dataset_id_to_contiguous_id = {1: 0}  # COCO ID 1 → internal ID 0
)

In [ ]:
pano_meta = MetadataCatalog.get("Pano360_scale_train").set(
    json_file=coco_keypoints_path, image_root=coco_images_root_path, evaluator_type="coco", **coco_meta,
    thing_dataset_id_to_contiguous_id = {1: 0}  # COCO ID 1 → internal ID 0
)
pano_meta = MetadataCatalog.get("Pano360_train").set(
    json_file=coco_keypoints_path, image_root=coco_images_root_path, evaluator_type="coco", **coco_meta,
    thing_dataset_id_to_contiguous_id = {1: 0}  # COCO ID 1 → internal ID 0
)

# Use Both

In [ ]:
from detectron2.data.datasets.coco_scale import COCOScale2017Calib
from detectron2.data.build import get_detection_dataset_dicts
coco_scale_calib_dataset = COCOScale2017Calib(
    train_calib,
    get_detection_dataset_dicts(
        coco_scale_dataset_name, True, 2, None, check_consistency=True
    ),
)
coco_scale_calib_dataset_name = "COCOScale2017Calib_train"
if coco_scale_calib_dataset_name in DatasetCatalog:
    DatasetCatalog.remove(coco_scale_calib_dataset_name)
DatasetCatalog.register(coco_scale_calib_dataset_name, coco_scale_calib_dataset)
meta = MetadataCatalog.get(coco_scale_calib_dataset_name).set(
    json_file=coco_keypoints_path, image_root=coco_images_root_path, evaluator_type="coco", **coco_meta,
    thing_dataset_id_to_contiguous_id = {1: 0}  # COCO ID 1 → internal ID 0
)

# Train

In [ ]:
import os
from detectron2.engine import HybridScaleTrainer
from detectron2 import model_zoo
from detectron2.config import get_cfg

cfg = get_cfg()
config_path = "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml"
cfg.merge_from_file(model_zoo.get_config_file(config_path))
experiment_name = "coco-scale-roih-biasl2-hflip"
# cfg.MODEL.CAMERA_HEAD.NUM_CONV = 0
# cfg.MODEL.CAMERA_HEAD.NUM_FC = 1
cfg.OUTPUT_DIR = os.path.join("output", experiment_name)
#cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_path)  # Let training initialize from model zoo
cfg.MODEL.WEIGHTS = os.path.join("output", experiment_name, "model_final.pth")
# IMS_PER_BATCH: 4 (No Height + AMP) - 3 (Height + AMP) - 2 (Height + Refine + AMP) - (2 Height + Refine)
cfg.SOLVER.IMS_PER_BATCH = 2 # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
cfg.MODEL.KEYPOINT_ON = False
cfg.MODEL.HEIGHT_ON = True 
cfg.MODEL.POINT_NET_ON = True 
cfg.MODEL.HEIGHT_REFINE_ON = True
cfg.MODEL.META_ARCHITECTURE = "GeneralizedCamRCNN"
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # Number of classes
cfg.MODEL.ROI_KEYPOINT_HEAD.NUM_KEYPOINTS = 17 # Number of keypoints
cfg.MODEL.ROI_HEADS.NAME = "HeightStandardROIHeads"
cfg.MODEL.ROI_KEYPOINT_HEAD.NAME = "KRCNNConvDeconvUpsampleHead"
cfg.VIS_PERIOD = 10
cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS = False  # Dataset is filtered before entering MixedDataset with build_detection_dataset.
cfg.DATALOADER.ASPECT_RATIO_GROUPING = False  # Doesn't work at the current implementation of MixedDataset
cfg.SOLVER.MAX_ITER = 5000
# cfg.SOLVER.WARMUP_ITERS = 0
cfg.SOLVER.AMP.ENABLED = False  # Enable AMP here -- improve 10s per iter approx.
cfg.SOLVER.RATIO_PANO360 = (3, 1)
cfg.FLOAT32_PRECISION = "highest"
# cfg.DATALOADER
cfg.DATASETS.TRAIN = (coco_scale_calib_dataset_name, )
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 4

# SVMIW Losses
cfg.MODEL.ROI_KEYPOINT_HEAD.LOSS_WEIGHT = 10  # alpha_4
cfg.MODEL.ROI_BOX_HEAD.BBOX_REG_LOSS_WEIGHT = 10  # alpha_5
cfg.MODEL.HEIGHT_HEAD.LOSS_WEIGHT = 0.05  # alpha 2
cfg.MODEL.HEIGHT_HEAD.REDUCE_METHOD = "softmax"
cfg.MODEL.HEIGHT_HEAD.SMOOTH_L1_BETA = 0.1

cfg.MODEL.CAMERA_HEAD.LOSS_CRITERION = "softargmax_l2_biased"

cfg.MODEL.POINT_NET.POOLING="max"
cfg.MODEL.POINT_NET.TEMPERATURE = 1.0
# NOTE: Check these values
cfg.MODEL.POINT_NET.BN = False
cfg.MODEL.POINT_NET.TRANSFORM = True
cfg.MODEL.HEIGHT_REFINE_ON = True
cfg.MODEL.POINT_NET.DETACH = False
cfg.MODEL.POINT_NET.REFINE_TEMPERATURE = 1.0
cfg.MODEL.POINT_NET.REFINE_LAYERS = 2

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.6,max_split_size_mb:128,expandable_segments:True"
trainer = HybridScaleTrainer(cfg) 
trainer.resume_or_load(resume=False)
model = trainer.model
# experiment_name = "test-debug-calib-dataset"
# exp_weights_path = "./ugroutput/output/model_final.pth"

# def load_state_dict(exp_weights_path):
#     checkpoint = torch.load(exp_weights_path, map_location="cpu")
#     state_dict = checkpoint.get("model", checkpoint)
#     state_dict = {
#         k.replace("module.", ""): v
#         for k, v in state_dict.items()
#     }
#     return state_dict

# # calib_state_dict = load_state_dict(exp_weights_path)
# # coco_state_dict = load_state_dict(cfg.MODEL.WEIGHTS)
# # coco_state_dict = load_state_dict(cfg.MODEL.WEIGHTS)
# # Filter only camera_head weights
# calib_state_dict = {
#     k: v for k, v in calib_state_dict.items() if "camera_head" in k
# }
# # Merge both state dicts to have the full state dict to load. Make sure the argument is filtered. 
# coco_state_dict.update(calib_state_dict)
# # Load into trainer.model
# missing, unexpected = model.load_state_dict(coco_state_dict, strict=False)
# print("Loaded camera_head weights into trainer.model")
# print("Missing keys:", missing)
# print("Unexpected keys:", unexpected)
# trainer.train()

In [ ]:
# from detectron2.data import build_detection_test_loader
# from detectron2.data.dataset_mapper import COCOScaleMapper
# model = trainer.model
# dataloader = build_detection_test_loader(
#     cfg,
#     "COCOScale2017_train",
#     mapper=COCOScaleMapper(cfg, is_train=False),
#     batch_size=4,
# )
# dataloader = iter(dataloader)

In [ ]:
# batched_inputs = next(dataloader)
# model.eval()
# with torch.no_grad():
#     # NOTE: Set do_postprocess to false if we use visualization using batched_inputs
#     # Since dataset_mapper resizes already the image.
#     instances, camrcnn_data = model.inference(batched_inputs, do_postprocess=False)

In [ ]:
# import matplotlib.pyplot as plt
# images = model.visualize_prediction(batched_inputs, instances, camrcnn_data)
# plt.figure(figsize=(16, 8))
# for img in images:
#     plt.imshow(img)
#     plt.axis("off")
#     plt.show()

# Pred Custom Dataset

In [ ]:
import json
from detectron2.structures import BoxMode

DATA_JSON_PATH = base_path / "data" / "mydataset.json"

class MyCustomData:
    def __init__(
        self,
        *,
        data_json_path: str, 
        debug: bool = False,
        debug_size: int = 100,
        shuffle: bool = False,
    ):
        with open(data_json_path, "r", encoding="utf-8") as file:
            self.data = json.load(file)
        
        if debug:
            self.data = self.data[:debug_size]
        if shuffle:
            random.shuffle(self.data)

    def __getitem__(self, k):
        im_path = self.data[k]["image_path"]
        image_id = self.data[k]["image_name"]
        instances = [dict(
            bbox=self.data[k]["gt_bbox"],
            bbox_mode=BoxMode.XYXY_ABS,
            category_id=0,
        )]
        output = self.data[k].copy()
        # output.pop("camera_height")
        return dict(
            source="custom",
            file_name=im_path,
            image_id=image_id,
            annotations=instances,
            **output,
        )

    def get_all_items(self):
        for i, _ in enumerate(self.data):
            yield self[i]

    def __call__(self):
        return self

    def __len__(self):
        return len(self.data)

debug = False
mycustom_dataset = MyCustomData(data_json_path=DATA_JSON_PATH, debug=debug, debug_size=100)

In [ ]:
mycustom_dataset_name = "MyCustomDataset"
if mycustom_dataset_name in DatasetCatalog:
    DatasetCatalog.remove(mycustom_dataset_name)
DatasetCatalog.register(mycustom_dataset_name, mycustom_dataset)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer

if debug:
    max_vis = 5
    for i, d in enumerate(mycustom_dataset):
        img = cv2.imread(d["file_name"])
        visualizer = Visualizer(img[:, :, ::-1], scale=0.5)
        visualizer.draw_dataset_dict(d)
        out = visualizer.get_output()
        img = out.get_image()
        plt.imshow(img)
        plt.show()
        if max_vis == i:
            break

In [ ]:
from detectron2.data import build_detection_test_loader
model = trainer.model
dataloader = build_detection_test_loader(
    cfg,
    mycustom_dataset_name,
    batch_size=4,
)

In [ ]:
import matplotlib.pyplot as plt
predictions = []
model.eval()
do_postprocess = True
max_vis = 5
vis = 0
with torch.no_grad():
    dataloader_iter = iter(dataloader)
    for batched_inputs in dataloader_iter:
    # NOTE: Set do_postprocess to false if we use visualization using batched_inputs
    # Since dataset_mapper resizes already the image.
        instances, camrcnn_data = model.inference(batched_inputs, do_postprocess=do_postprocess)
        for i, input in enumerate(batched_inputs):
            input["pred_height"] = instances[i].pred_height[0].cpu().item()
            input["pred_box"] = list(map(lambda x: int(x), instances[i].pred_boxes[0].tensor.cpu().numpy()[0]))
            input["pred_camera_height"] = camrcnn_data["yc_est"][0].cpu().item()
            input["vt_losses"] = [x.item() for x in camrcnn_data["vt_losses"]]
            input["vt_loss"] = sum(camrcnn_data["vt_losses"]) / len(camrcnn_data["vt_losses"])
            predictions.append({k: v for k, v in input.items() if k != "image"})
    if not do_postprocess and vis < max_vis:
        vis += 1
        images = model.visualize_prediction(batched_inputs, instances, camrcnn_data)
        for img in images:
            plt.imshow(img)
            plt.axis("off")
            plt.show()
    # else we need to use the original image to plot the bbox

In [ ]:
import pandas as pd
print(predictions)
results_df = pd.DataFrame(predictions)
display(results_df)

In [ ]:
print(results_df["pred_camera_height"].var())
print(results_df["pred_height"].var())
print(results_df["vt_loss"].mean())

In [ ]:
display(results_df["vt_losses"].apply(lambda x: np.asarray(x).argmax()).describe())
display(results_df["vt_losses"].apply(lambda x: np.asarray(x).argmin()).describe())

In [ ]:
results_df["error"] = abs(results_df["gt_height"] - results_df["pred_height"])
results_df["error"] = results_df["error"].astype(float)
results_stats_df = results_df[["camera", "error"]].groupby("camera").describe(percentiles=[0.5, 0.75])
display(results_stats_df)
# Flatten the columns
results_stats_df.columns = ['_'.join(col).strip() for col in results_stats_df.columns.values]
results_stats_df = results_stats_df.reset_index()
results_df = pd.merge(
    results_df,
    results_stats_df,
    on="camera",
)
with open(base_path / "data" / "results_table.tex", "w") as f:
    f.write(results_stats_df.to_latex(index=False, float_format="%.2f", label="tab:AutoResults", position="ht"))


In [ ]:
results_df.to_excel("data/preds.xlsx", index=False)

In [ ]:
def draw_bbox(image_path, bbox, color=(0, 255, 0), thickness=2, gt_label: str = None, pred_label: str = None):
    """
    Draw a bounding box on an image and display it inline (Jupyter Notebook).

    Parameters:
        image_path (str): Path to the image file
        bbox (list or tuple): [x1, y1, x2, y2]
        color (tuple): BGR color (default green)
        thickness (int): Rectangle thickness
    """
    # Read image (OpenCV loads in BGR)
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError("Could not load image. Check the path.")
    x1, y1, x2, y2 = bbox

    # Draw rectangle
    cv2.rectangle(image, (x1, y1), (x2, y2), color, thickness)

    # Convert BGR to RGB for matplotlib
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    if gt_label:
        cv2.putText(image_rgb, gt_label, (x1, y1), cv2.FONT_HERSHEY_DUPLEX, 4, (0, 255, 0), thickness=4)
    if pred_label:
        cv2.putText(image_rgb, pred_label, (x1, y2), cv2.FONT_HERSHEY_DUPLEX, 4, (255, 0, 0), thickness=4)

    # Display
    plt.figure(figsize=(8, 6))
    plt.imshow(image_rgb)
    plt.axis("off")
    plt.show()

In [ ]:
display(results_df)

In [ ]:
results_df["pred_box"] = results_df["pred_box"].apply(lambda x: [int(y) for y in x])

In [ ]:
max_error_df = results_df[results_df["error"] >= results_df["error_max"]].sort_values("image_name")
with pd.option_context("display.max_rows", 3, "display.max_columns", None):
    display(max_error_df)
for i in range(len(max_error_df)):
    # predictions_path = dataset_path / "mydataset_results.json"
    print(max_error_df.iloc[i]["camera"])
    print(max_error_df.iloc[i]["image_path"])
    draw_bbox(
        max_error_df.iloc[i]["image_path"],
        max_error_df.iloc[i]["pred_box"],
        gt_label=f'{max_error_df.iloc[i]["gt_height"]:.2f}',
        pred_label=f'{max_error_df.iloc[i]["pred_height"]:.2f}',
    )

In [ ]:
min_error_df = results_df[results_df["error"] <= results_df["error_min"]].sort_values("image_name")
with pd.option_context("display.max_rows", 3, "display.max_columns", None):
    display(min_error_df)
for i in range(len(min_error_df)):
    # predictions_path = dataset_path / "mydataset_results.json"
    print(min_error_df.iloc[i]["image_path"])
    print(min_error_df.iloc[i]["camera"])
    draw_bbox(
        min_error_df.iloc[i]["image_path"],
        min_error_df.iloc[i]["pred_box"],
        gt_label=f'{min_error_df.iloc[i]["gt_height"]:.2f}',
        pred_label=f'{min_error_df.iloc[i]["pred_height"]:.2f}',
    )

In [ ]:
results_df["camera_error"] = abs(results_df["camera_height"] - results_df["pred_camera_height"])
results_df["camera_error"] = results_df["camera_error"].astype(float)
results_stats_df = results_df[["camera", "camera_error"]].groupby("camera").describe(percentiles=[0.5, 0.75])
display(results_stats_df)
# Flatten the columns
results_stats_df.columns = ['_'.join(col).strip() for col in results_stats_df.columns.values]
results_stats_df = results_stats_df.reset_index()
results_df = pd.merge(
    results_df,
    results_stats_df,
    on="camera",
    suffixes=("", "_camera")
)
with open(base_path / "data" / "cam_results_table.tex", "w") as f:
    f.write(results_stats_df.to_latex(index=False, float_format="%.2f", label="tab:AutoResults", position="ht"))
display(results_df)

In [ ]:
results_df.to_excel("data/cam_preds.xlsx", index=False)

# Pred Val COCOSCale

In [ ]:
cfg.DATASETS.TEST = (coco_scale_val_dataset_name, )
trainer.test(cfg, trainer.model)

In [ ]:
# vt_loss_val=0.08476053816893002

In [ ]:
cfg.DATASETS.TEST = (coco_scale_test_dataset_name, )
trainer.test(cfg, trainer.model)

In [ ]:
# 'vt_loss_val' = 0.08108685230525256

# KITTY Eval

In [ ]:
from detectron2.data.datasets.coco_scale import KITTICocoDataset
KITTY_TRAIN_NAME = "Kitty_train"
KITTY_ROOT = os.path.join("data", "Kitty")
KITTY_IMAGE_DIR = os.path.join(KITTY_ROOT, "data_object_image_2", "training", "image_2")
KITTY_LABEL_DIR = os.path.join(KITTY_ROOT, "data_object_label_2", "training", "label_2")
KITTY_OUTPUT_JSON = os.path.join("data", "Kitty", "kitti_coco.json")
if KITTY_TRAIN_NAME in DatasetCatalog:
    DatasetCatalog.remove(KITTY_TRAIN_NAME)
DatasetCatalog.register(
    KITTY_TRAIN_NAME,
    KITTICocoDataset(
        KITTY_OUTPUT_JSON,
        KITTY_IMAGE_DIR,
        debug=debug, 
    )
)
coco_meta = _get_builtin_metadata("coco")
MetadataCatalog.get(KITTY_TRAIN_NAME).set(
    thing_dataset_id_to_contiguous_id={1: 0, 3: 1}  # COCO ID 1 → internal ID 0
)

In [ ]:
cfg.DATASETS.TEST = (KITTY_TRAIN_NAME, )
trainer.test(cfg, trainer.model)

In [ ]:
# {'kitty_height_mae': 0.10331526950156028,
#  'kitty_num_matches': 2221,
#  'kitty_num_samples': 1287,
#  'kitty_vt_loss_mean': 0.07338283392509865,
#  'kitty_num_invalid_samples': 93,
#  'kitty_num_empty_samples': 65}